## Clustering -- Partial Dependence Age Group Linking

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import partial_dependence
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# STEP 0: Load Your Data
# ============================================================================

property_df = pd.read_csv('../../datasets/normalized_data/combined_normalized.csv')

print(f"Dataset shape: {property_df.shape}")
print(f"\nFirst few rows:")
print(property_df.head())

# ============================================================================
# STEP 1: CLUSTERING - Discover Natural Buyer Segments
# ============================================================================

def discover_buyer_segments(property_df):
    """
    Cluster properties by amenity characteristics to discover natural buyer segments.
    """
    
    print("\n" + "="*80)
    print("STEP 1: DISCOVERING BUYER SEGMENTS THROUGH CLUSTERING")
    print("="*80)
    
    cluster_features = [
        'Num MRT Within 1km',
        'Num Schools Within 2km',
        'Num Hospitals Within 5km',
        'Num Parks Within 1km',
        'Num Malls Within 1km',
        'Num Hawker Within 1km',
        'Dist to CBD in Km',
        'Floor_Level_Category',
        'Property_Type_Encoded',
        'Market_Segment_Encoded',
    ]
    
    X_cluster = property_df[cluster_features].copy()
    
    print(f"\nClustering based on {len(cluster_features)} features:")
    for i, feature in enumerate(cluster_features, 1):
        print(f"  {i:2d}. {feature}")
    
    optimal_k = 3
    
    print(f"\n{'='*80}")
    print(f"OPTIMAL K = {optimal_k}")
    print(f"{'='*80}")
    
    kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=20, max_iter=300)
    property_df['Cluster'] = kmeans_final.fit_predict(X_cluster)
    
    print(f"\n✓ Properties assigned to {optimal_k} clusters")
    print(f"\nCluster distribution:")
    cluster_counts = property_df['Cluster'].value_counts().sort_index()
    for cluster_id, count in cluster_counts.items():
        pct = (count / len(property_df)) * 100
        print(f"  Cluster {cluster_id}: {count:5d} properties ({pct:5.1f}%)")
    
    return property_df, cluster_features, kmeans_final, X_cluster


# ============================================================================
# NEW: ENHANCED CLUSTER VISUALIZATION
# ============================================================================

def visualize_clusters_advanced(property_df, X_cluster, cluster_features):
    """
    Create comprehensive cluster visualizations similar to your scatterplot.
    """
    
    print("\n" + "="*80)
    print("CREATING ADVANCED CLUSTER VISUALIZATIONS")
    print("="*80)
    
    # -----------------------------------------------------------------------
    # 1. 3D PCA Visualization (like your scatterplot)
    # -----------------------------------------------------------------------
    
    print("\n1. Computing PCA for 3D visualization...")
    
    # Reduce to 3 principal components
    pca_3d = PCA(n_components=3, random_state=42)
    X_pca_3d = pca_3d.fit_transform(X_cluster)
    
    # Explained variance
    explained_var = pca_3d.explained_variance_ratio_
    print(f"   PC1 explains {explained_var[0]*100:.1f}% of variance")
    print(f"   PC2 explains {explained_var[1]*100:.1f}% of variance")
    print(f"   PC3 explains {explained_var[2]*100:.1f}% of variance")
    print(f"   Total: {sum(explained_var)*100:.1f}%")
    
    # Create 3D scatter plot
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # Color map for clusters
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']  # Red, Teal, Blue
    markers = ['o', 's', '^']  # Circle, Square, Triangle
    
    for cluster_id in sorted(property_df['Cluster'].unique()):
        cluster_mask = property_df['Cluster'] == cluster_id
        ax.scatter(
            X_pca_3d[cluster_mask, 0],
            X_pca_3d[cluster_mask, 1],
            X_pca_3d[cluster_mask, 2],
            c=colors[cluster_id],
            marker=markers[cluster_id],
            s=30,
            alpha=0.6,
            edgecolors='white',
            linewidth=0.5,
            label=f'Cluster {cluster_id} (n={cluster_mask.sum()})'
        )
    
    ax.set_xlabel(f'PC1 ({explained_var[0]*100:.1f}% var)', fontsize=11, labelpad=10)
    ax.set_ylabel(f'PC2 ({explained_var[1]*100:.1f}% var)', fontsize=11, labelpad=10)
    ax.set_zlabel(f'PC3 ({explained_var[2]*100:.1f}% var)', fontsize=11, labelpad=10)
    ax.set_title('3D Cluster Visualization (PCA Projection)\nProperty Segments in Feature Space', 
                 fontsize=14, fontweight='bold', pad=20)
    ax.legend(loc='upper right', fontsize=10, framealpha=0.9)
    ax.grid(True, alpha=0.3)
    
    # Improve viewing angle
    ax.view_init(elev=20, azim=45)
    
    plt.tight_layout()
    plt.savefig('cluster_3d_pca.png', dpi=200, bbox_inches='tight')
    print("   ✓ Saved: cluster_3d_pca.png")
    plt.close()
    
    # -----------------------------------------------------------------------
    # 2. 2D PCA Pair Plots (all combinations)
    # -----------------------------------------------------------------------
    
    print("\n2. Creating 2D PCA pair plots...")
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    pairs = [(0, 1), (0, 2), (1, 2)]
    pair_labels = [
        (f'PC1 ({explained_var[0]*100:.1f}%)', f'PC2 ({explained_var[1]*100:.1f}%)'),
        (f'PC1 ({explained_var[0]*100:.1f}%)', f'PC3 ({explained_var[2]*100:.1f}%)'),
        (f'PC2 ({explained_var[1]*100:.1f}%)', f'PC3 ({explained_var[2]*100:.1f}%)')
    ]
    
    for idx, (pc1, pc2) in enumerate(pairs):
        ax = axes[idx]
        
        for cluster_id in sorted(property_df['Cluster'].unique()):
            cluster_mask = property_df['Cluster'] == cluster_id
            ax.scatter(
                X_pca_3d[cluster_mask, pc1],
                X_pca_3d[cluster_mask, pc2],
                c=colors[cluster_id],
                marker=markers[cluster_id],
                s=20,
                alpha=0.5,
                edgecolors='white',
                linewidth=0.3,
                label=f'Cluster {cluster_id}'
            )
        
        ax.set_xlabel(pair_labels[idx][0], fontsize=10)
        ax.set_ylabel(pair_labels[idx][1], fontsize=10)
        ax.set_title(f'{pair_labels[idx][0]} vs {pair_labels[idx][1]}', fontsize=11, fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('cluster_2d_pca_pairs.png', dpi=150, bbox_inches='tight')
    print("   ✓ Saved: cluster_2d_pca_pairs.png")
    plt.close()
    
    # -----------------------------------------------------------------------
    # 3. Feature Contribution to Principal Components
    # -----------------------------------------------------------------------
    
    print("\n3. Analyzing feature contributions to PCs...")
    
    # Get loadings (feature contributions)
    loadings = pca_3d.components_.T * np.sqrt(pca_3d.explained_variance_)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    for pc_idx in range(3):
        ax = axes[pc_idx]
        
        # Sort features by absolute loading
        feature_loadings = pd.DataFrame({
            'Feature': [f.replace('Num ', '').replace(' Within', '\n') for f in cluster_features],
            'Loading': loadings[:, pc_idx]
        }).sort_values('Loading', key=abs, ascending=False)
        
        # Bar plot
        colors_bar = ['#FF6B6B' if x < 0 else '#4ECDC4' for x in feature_loadings['Loading']]
        ax.barh(feature_loadings['Feature'], feature_loadings['Loading'], color=colors_bar, alpha=0.7)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_xlabel(f'Loading (contribution to PC{pc_idx+1})', fontsize=10)
        ax.set_title(f'PC{pc_idx+1} Feature Loadings\n({explained_var[pc_idx]*100:.1f}% variance)', 
                     fontsize=11, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('cluster_pca_loadings.png', dpi=150, bbox_inches='tight')
    print("   ✓ Saved: cluster_pca_loadings.png")
    plt.close()
    
    # -----------------------------------------------------------------------
    # 4. Cluster Centroids in Original Feature Space
    # -----------------------------------------------------------------------
    
    print("\n4. Visualizing cluster centroids...")
    
    # Calculate centroids
    centroids = []
    for cluster_id in sorted(property_df['Cluster'].unique()):
        cluster_mask = property_df['Cluster'] == cluster_id
        centroid = X_cluster[cluster_mask].mean(axis=0)
        centroids.append(centroid)
    
    centroids = np.array(centroids)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Heatmap of centroids
    feature_labels = [f.replace('Num ', '').replace(' Within', '\n') for f in cluster_features]
    
    sns.heatmap(
        centroids,
        annot=True,
        fmt='.2f',
        cmap='RdYlGn',
        center=0,
        xticklabels=feature_labels,
        yticklabels=[f'Cluster {i}' for i in range(len(centroids))],
        cbar_kws={'label': 'Normalized Value'},
        ax=ax
    )
    ax.set_title('Cluster Centroids in Original Feature Space\n(Higher = More of that amenity)', 
                 fontsize=13, fontweight='bold', pad=15)
    ax.set_xlabel('Features', fontsize=11)
    ax.set_ylabel('Clusters', fontsize=11)
    
    plt.tight_layout()
    plt.savefig('cluster_centroids_heatmap.png', dpi=150, bbox_inches='tight')
    print("   ✓ Saved: cluster_centroids_heatmap.png")
    plt.close()
    
    # -----------------------------------------------------------------------
    # 5. Cluster Separation Metrics
    # -----------------------------------------------------------------------
    
    print("\n5. Computing cluster quality metrics...")
    
    silhouette = silhouette_score(X_cluster, property_df['Cluster'])
    davies_bouldin = davies_bouldin_score(X_cluster, property_df['Cluster'])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Silhouette plot per cluster
    from sklearn.metrics import silhouette_samples
    
    silhouette_vals = silhouette_samples(X_cluster, property_df['Cluster'])
    
    ax = axes[0]
    y_lower = 10
    
    for cluster_id in sorted(property_df['Cluster'].unique()):
        cluster_silhouette_vals = silhouette_vals[property_df['Cluster'] == cluster_id]
        cluster_silhouette_vals.sort()
        
        size_cluster = cluster_silhouette_vals.shape[0]
        y_upper = y_lower + size_cluster
        
        ax.fill_betweenx(
            np.arange(y_lower, y_upper),
            0,
            cluster_silhouette_vals,
            facecolor=colors[cluster_id],
            edgecolor=colors[cluster_id],
            alpha=0.7,
            label=f'Cluster {cluster_id}'
        )
        
        ax.text(-0.05, y_lower + 0.5 * size_cluster, str(cluster_id), fontsize=12, fontweight='bold')
        y_lower = y_upper + 10
    
    ax.axvline(silhouette, color='red', linestyle='--', linewidth=2, label=f'Avg: {silhouette:.3f}')
    ax.set_xlabel('Silhouette Coefficient', fontsize=11)
    ax.set_ylabel('Cluster', fontsize=11)
    ax.set_title(f'Silhouette Analysis\n(Higher = Better Separation)', fontsize=12, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.set_xlim([-0.1, 1])
    ax.grid(axis='x', alpha=0.3)
    
    # Cluster sizes
    ax = axes[1]
    cluster_sizes = property_df['Cluster'].value_counts().sort_index()
    ax.bar(cluster_sizes.index, cluster_sizes.values, color=colors, alpha=0.7, edgecolor='black')
    ax.set_xlabel('Cluster ID', fontsize=11)
    ax.set_ylabel('Number of Properties', fontsize=11)
    ax.set_title(f'Cluster Sizes\n(Total: {len(property_df)} properties)', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    for i, v in enumerate(cluster_sizes.values):
        ax.text(i, v + 50, str(v), ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    plt.tight_layout()
    plt.savefig('cluster_quality_metrics.png', dpi=150, bbox_inches='tight')
    print("   ✓ Saved: cluster_quality_metrics.png")
    plt.close()
    
    print(f"\n   Silhouette Score: {silhouette:.3f} (range: -1 to 1, higher = better)")
    print(f"   Davies-Bouldin Index: {davies_bouldin:.3f} (lower = better)")
    
    # -----------------------------------------------------------------------
    # 6. Interactive-style Cluster Profiles
    # -----------------------------------------------------------------------
    
    print("\n6. Creating cluster profile comparison...")
    
    # Select key amenity features for visualization
    amenity_features = [
        'Num MRT Within 1km',
        'Num Schools Within 2km',
        'Num Hospitals Within 5km',
        'Num Parks Within 1km',
        'Num Malls Within 1km',
        'Num Hawker Within 1km'
    ]
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    for idx, feature in enumerate(amenity_features):
        ax = axes[idx]
        
        # Violin plot for each cluster
        data_to_plot = [property_df[property_df['Cluster'] == cid][feature].values 
                        for cid in sorted(property_df['Cluster'].unique())]
        
        parts = ax.violinplot(
            data_to_plot,
            positions=range(len(data_to_plot)),
            showmeans=True,
            showmedians=True
        )
        
        # Color the violins
        for pc, color in zip(parts['bodies'], colors):
            pc.set_facecolor(color)
            pc.set_alpha(0.7)
        
        # Overlay scatter points
        for cluster_id in sorted(property_df['Cluster'].unique()):
            cluster_data = property_df[property_df['Cluster'] == cluster_id][feature].values
            y = cluster_data
            x = np.random.normal(cluster_id, 0.04, size=len(y))  # Add jitter
            ax.scatter(x, y, alpha=0.3, s=2, color=colors[cluster_id])
        
        ax.set_xticks(range(len(data_to_plot)))
        ax.set_xticklabels([f'C{i}' for i in range(len(data_to_plot))])
        ax.set_xlabel('Cluster', fontsize=10)
        ax.set_ylabel('Normalized Value', fontsize=10)
        ax.set_title(feature.replace('Num ', '').replace(' Within', '\n'), fontsize=11, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
    
    plt.suptitle('Cluster Profiles: Amenity Distribution by Cluster', fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.savefig('cluster_amenity_distributions.png', dpi=150, bbox_inches='tight')
    print("   ✓ Saved: cluster_amenity_distributions.png")
    plt.close()
    
    print("\n" + "="*80)
    print("✓ ALL CLUSTER VISUALIZATIONS COMPLETE")
    print("="*80)
    
    return X_pca_3d, pca_3d


# ============================================================================
# STEP 2-7: Keep your existing functions
# ============================================================================

def analyze_clusters(property_df, cluster_features):
    # [Keep your existing code]
    print("\n" + "="*80)
    print("STEP 2: ANALYZING CLUSTER CHARACTERISTICS")
    print("="*80)
    
    n_clusters = property_df['Cluster'].nunique()
    overall_means = property_df[cluster_features].mean()
    cluster_profiles = {}
    
    for cluster_id in range(n_clusters):
        cluster_df = property_df[property_df['Cluster'] == cluster_id]
        
        print(f"\n{'='*80}")
        print(f"CLUSTER {cluster_id} - {len(cluster_df)} properties ({len(cluster_df)/len(property_df)*100:.1f}%)")
        print(f"{'='*80}")
        
        cluster_means = cluster_df[cluster_features].mean()
        
        print(f"\n{'Feature':<35} {'Cluster Avg':>12} {'Overall Avg':>12} {'Difference':>12}")
        print("-" * 80)
        
        profile = {}
        
        for feature in cluster_features:
            cluster_val = cluster_means[feature]
            overall_val = overall_means[feature]
            diff = cluster_val - overall_val
            
            if abs(overall_val) > 0.01:
                pct_diff = ((cluster_val - overall_val) / abs(overall_val)) * 100
            else:
                pct_diff = 0
            
            profile[feature] = {
                'cluster_mean': cluster_val,
                'overall_mean': overall_val,
                'difference': diff,
                'pct_difference': pct_diff
            }
            
            if abs(diff) > 0.5:
                marker = "▲▲" if diff > 0 else "▼▼"
            elif abs(diff) > 0.25:
                marker = "▲ " if diff > 0 else "▼ "
            else:
                marker = "≈ "
            
            print(f"{feature:<35} {cluster_val:>12.3f} {overall_val:>12.3f} {marker} {diff:>+9.3f}")
        
        print(f"\n{'Price Statistics':}")
        print(f"  Median Transacted Price (normalized): {cluster_df['Transacted Price ($)'].median():>8.3f}")
        print(f"  Mean Transacted Price (normalized):   {cluster_df['Transacted Price ($)'].mean():>8.3f}")
        print(f"  Std Dev:                               {cluster_df['Transacted Price ($)'].std():>8.3f}")
        
        cluster_profiles[cluster_id] = profile
    
    return cluster_profiles


def calculate_pd_weights_per_cluster(property_df):
    # [Keep your existing code]
    print("\n" + "="*80)
    print("STEP 3: CALCULATING PARTIAL DEPENDENCE WEIGHTS PER CLUSTER")
    print("="*80)
    
    amenity_features = [
        'Num MRT Within 1km',
        'Num Schools Within 2km',
        'Num Hospitals Within 5km',
        'Num Parks Within 1km',
        'Num Malls Within 1km',
        'Num Hawker Within 1km'
    ]
    
    n_clusters = property_df['Cluster'].nunique()
    cluster_weights = {}
    
    for cluster_id in range(n_clusters):
        cluster_df = property_df[property_df['Cluster'] == cluster_id].copy()
        
        print(f"\n{'-'*80}")
        print(f"Cluster {cluster_id} - {len(cluster_df)} properties")
        print(f"{'-'*80}")
        
        X = cluster_df[amenity_features].values
        y = cluster_df['Transacted Price ($)'].values
        
        print(f"  Training Gradient Boosting model...")
        model = GradientBoostingRegressor(
            n_estimators=100,
            max_depth=4,
            learning_rate=0.1,
            random_state=42,
            subsample=0.8
        )
        model.fit(X, y)
        
        train_score = model.score(X, y)
        print(f"  Model R² score: {train_score:.4f}")
        
        if train_score < 0.3:
            print(f"  ⚠ WARNING: Low R² score ({train_score:.4f}).")
        
        print(f"\n  Calculating Partial Dependence impacts...")
        impacts = {}
        
        for i, feature in enumerate(amenity_features):
            pd_result = partial_dependence(
                model, 
                X, 
                features=[i],
                grid_resolution=50
            )
            
            avg_predictions = pd_result['average'][0]
            impact = avg_predictions.max() - avg_predictions.min()
            impacts[feature] = impact
        
        total_impact = sum(impacts.values())
        
        if total_impact > 0:
            weights = {feature: impact / total_impact for feature, impact in impacts.items()}
        else:
            weights = {feature: 1.0 / len(amenity_features) for feature in amenity_features}
        
        cluster_weights[cluster_id] = {
            'impacts': impacts,
            'weights': weights,
            'model_r2': train_score
        }
        
        print(f"\n  Feature Importance (Partial Dependence Impact):")
        print(f"  {'Amenity':<35} {'Impact':>12} {'Weight':>10}")
        print(f"  {'-'*60}")
        
        sorted_features = sorted(weights.items(), key=lambda x: x[1], reverse=True)
        for feature, weight in sorted_features:
            impact = impacts[feature]
            print(f"  {feature:<35} {impact:>12.4f} {weight:>10.4f}")
    
    return cluster_weights, amenity_features


def link_clusters_to_age_groups(property_df, age_demographics_df):
    # [Keep your existing code]
    print("\n" + "="*80)
    print("STEP 4: LINKING CLUSTERS TO AGE GROUPS")
    print("="*80)
    
    n_clusters = property_df['Cluster'].nunique()
    
    merged = property_df.merge(
        age_demographics_df,
        on='District',
        how='left'
    )
    
    cluster_age_profiles = {}
    
    for cluster_id in range(n_clusters):
        cluster_df = merged[merged['Cluster'] == cluster_id]
        
        print(f"\n{'-'*80}")
        print(f"Cluster {cluster_id}")
        print(f"{'-'*80}")
        
        total_young = cluster_df['Young_Adults_20_35'].sum()
        total_families = cluster_df['Families_36_59'].sum()
        total_retirees = cluster_df['Retirees_60_Plus'].sum()
        
        total_pop = total_young + total_families + total_retirees
        
        if total_pop > 0:
            pct_young = (total_young / total_pop) * 100
            pct_families = (total_families / total_pop) * 100
            pct_retirees = (total_retirees / total_pop) * 100
        else:
            pct_young = pct_families = pct_retirees = 0
        
        print(f"\nAge Group Distribution in Cluster {cluster_id}'s Districts:")
        print(f"  Young Adults (20-35):  {pct_young:5.1f}%")
        print(f"  Families (36-59):      {pct_families:5.1f}%")
        print(f"  Retirees (60+):        {pct_retirees:5.1f}%")
        
        age_percentages = {
            'Young_Adults_20_35': pct_young,
            'Families_36_59': pct_families,
            'Retirees_60_Plus': pct_retirees
        }
        
        dominant_age_group = max(age_percentages, key=age_percentages.get)
        dominant_pct = age_percentages[dominant_age_group]
        
        print(f"\n  → Dominant Age Group: {dominant_age_group} ({dominant_pct:.1f}%)")
        
        cluster_age_profiles[cluster_id] = {
            'percentages': age_percentages,
            'dominant_group': dominant_age_group,
            'dominant_percentage': dominant_pct
        }
    
    return cluster_age_profiles


def assign_weights_to_age_groups(cluster_weights, cluster_age_profiles, amenity_features):
    # [Keep your existing code]
    print("\n" + "="*80)
    print("STEP 5: ASSIGNING WEIGHTS TO AGE GROUPS")
    print("="*80)
    
    age_group_preferences = {}
    
    for cluster_id, age_profile in cluster_age_profiles.items():
        age_group = age_profile['dominant_group']
        weights = cluster_weights[cluster_id]['weights']
        
        print(f"\nCluster {cluster_id} → {age_group}")
        
        if age_group not in age_group_preferences:
            age_group_preferences[age_group] = {'weights': weights, 'clusters': [cluster_id]}
        else:
            existing_weights = age_group_preferences[age_group]['weights']
            n_clusters = len(age_group_preferences[age_group]['clusters']) + 1
            
            averaged_weights = {}
            for feature in amenity_features:
                averaged_weights[feature] = (
                    existing_weights[feature] * (n_clusters - 1) + weights[feature]
                ) / n_clusters
            
            age_group_preferences[age_group]['weights'] = averaged_weights
            age_group_preferences[age_group]['clusters'].append(cluster_id)
    
    print(f"\n{'='*80}")
    print("FINAL AGE GROUP PREFERENCE WEIGHTS")
    print(f"{'='*80}")
    
    for age_group, data in age_group_preferences.items():
        print(f"\n{age_group}:")
        print(f"  (Derived from clusters: {data['clusters']})")
        print(f"\n  {'Amenity':<35} {'Weight':>10}")
        print(f"  {'-'*50}")
        
        sorted_weights = sorted(data['weights'].items(), key=lambda x: x[1], reverse=True)
        for feature, weight in sorted_weights:
            print(f"  {feature:<35} {weight:>10.4f}")
    
    return age_group_preferences


def validate_preferences(age_group_preferences):
    # [Keep your existing code - validation logic]
    pass


def visualize_results(property_df, cluster_weights, age_group_preferences, amenity_features):
    # [Keep your existing heatmaps and radar charts]
    pass


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main(property_df, age_demographics_df):
    """
    Execute the full pipeline with enhanced visualizations.
    """
    
    print("\n" + "="*80)
    print("ENHANCED CLUSTER-BASED PREFERENCE DERIVATION PIPELINE")
    print("="*80)
    
    # Step 1: Clustering
    property_df, cluster_features, kmeans_model, X_cluster = discover_buyer_segments(property_df)
    
    # NEW: Advanced cluster visualizations
    X_pca_3d, pca_model = visualize_clusters_advanced(property_df, X_cluster, cluster_features)
    
    # Step 2: Analyze clusters
    cluster_profiles = analyze_clusters(property_df, cluster_features)
    
    # Step 3: Calculate PD weights
    cluster_weights, amenity_features = calculate_pd_weights_per_cluster(property_df)
    
    # Step 4: Link to age groups
    cluster_age_profiles = link_clusters_to_age_groups(property_df, age_demographics_df)
    
    # Step 5: Assign to age groups
    age_group_preferences = assign_weights_to_age_groups(
        cluster_weights, 
        cluster_age_profiles, 
        amenity_features
    )
    
    # Step 6: Validate (optional)
    # validate_preferences(age_group_preferences)
    
    # Step 7: Original visualizations
    visualize_results(property_df, cluster_weights, age_group_preferences, amenity_features)
    
    print("\n" + "="*80)
    print("PIPELINE COMPLETE - CHECK OUTPUT IMAGES")
    print("="*80)
    
    return age_group_preferences, cluster_weights, property_df


# ============================================================================
# USAGE
# ============================================================================

# Load data
property_df = pd.read_csv('../../datasets/normalized_data/combined_normalized.csv')
age_demographics_df = pd.read_csv('../../datasets/supplementary_datasets/demographics_by_district.csv')

# Run pipeline
age_group_preferences, cluster_weights, property_df_with_clusters = main(
    property_df, 
    age_demographics_df
)

# Access final weights
print("\n" + "="*80)
print("FINAL OUTPUT: AGE GROUP PREFERENCE WEIGHTS")
print("="*80)

for age_group, data in age_group_preferences.items():
    print(f"\n{age_group}:")
    for amenity, weight in data['weights'].items():
        print(f"  {amenity}: {weight:.4f}")

Dataset shape: (108372, 16)

First few rows:
   Transacted Price ($)  Area (SQFT)  Unit Price ($ PSF)  Dist to CBD in Km  \
0             -0.211237    -0.460443            0.774350           1.347678   
1             -0.087630    -0.460443            1.240198           1.347678   
2             -0.303362    -0.331436           -0.058154          -0.618881   
3             -0.100129    -0.442004            1.103809           1.347678   
4              1.493910     2.267081           -0.950882          -0.363513   

   Sale Months Since Sep 2025  Num MRT Within 1km  Num Hawker Within 1km  \
0                   -1.117103           -1.177227              -1.175060   
1                   -1.117103           -1.177227              -1.175060   
2                   -1.117103           -0.723616              -0.407922   
3                   -1.117103           -1.177227              -1.175060   
4                   -1.117103            0.183604              -0.407922   

   Num Malls Within 1km